# 11. LangSmith Prompt Management - SDK로 프롬프트 관리하기

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `client.push_prompt()`로 ChatPromptTemplate을 LangSmith Hub에 업로드할 수 있어요
2. `client.pull_prompt()`로 최신 버전, 특정 태그, 특정 커밋을 각각 불러올 수 있어요
3. 모델을 포함한 체인을 push/pull하여 원클릭으로 배포 준비를 할 수 있어요
4. F-string과 Mustache 템플릿 형식의 차이를 이해하고 적절하게 선택할 수 있어요
5. `staging` → `production` 태그를 활용한 환경별 프롬프트 관리를 구축할 수 있어요

## 사전 지식

- 10-Playground: Playground UI에서 프롬프트 저장/커밋/태그 개념
- ChatPromptTemplate, ChatOpenAI, LCEL 체인 구성 기초

## Prompt Management 개요

LangSmith Prompt Management는 **프롬프트를 중앙에서 버전 관리**하는 시스템이에요.
Git이 코드를 관리하듯, LangSmith가 프롬프트를 관리해요.

```mermaid
flowchart LR
    subgraph 개발환경
        A["ChatPromptTemplate<br/>생성"] --> B["push_prompt()<br/>Hub에 업로드"]
    end
    
    subgraph LangSmith Hub
        C["버전 1 (커밋 a3f9)"] 
        D["버전 2 (커밋 b7d2)"] 
        E["버전 3 (커밋 c9e8)"] 
        C --> D --> E
        F["태그: staging → b7d2"]
        G["태그: production → a3f9"]
    end
    
    subgraph 운영환경
        H["pull_prompt(:latest)<br/>최신 버전"] 
        I["pull_prompt(:production)<br/>안정 버전"]
        J["pull_prompt(:staging)<br/>테스트 버전"]
    end
    
    B --> C
    B --> D
    B --> E
    E --> H
    G --> I
    F --> J

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class A,B input
    class C,D,E,F,G storage
    class H,I,J output
```

### Prompt Management의 핵심 구성요소

| 구성요소 | 설명 | 예시 |
|---------|------|-----|
| **Prompt** | 저장된 프롬프트 템플릿 | `joke-generator` |
| **Commit** | 자동 생성되는 불변 버전 해시 | `a3f92c1d` |
| **Tag** | 커밋을 가리키는 사람이 읽기 쉬운 별명 | `production`, `staging`, `v1` |
| **Owner** | 프롬프트 소유자 (개인 또는 조직) | `efriis/my-prompt` |

> 🔑 **핵심 개념**: **커밋 해시**는 변경되지 않는 고유 식별자이고, **태그**는 그 해시를 가리키는 포인터예요. 태그는 새 커밋으로 재지정할 수 있어서, 코드에서 `pull_prompt("name:production")`은 항상 현재 production으로 지정된 버전을 가져와요.

## 환경 설정

In [ ]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
# .env 파일에서 OPENAI_API_KEY, LANGSMITH_API_KEY를 로드해요
from dotenv import load_dotenv

load_dotenv()
print("환경 변수 로드 완료")

In [ ]:
# ---------------------------------------------------
# LangSmith 클라이언트 초기화 + 추적 설정
# ---------------------------------------------------
import os
from langsmith import Client

os.environ["LANGSMITH_PROJECT"] = "Ch11-Prompt-Management"

# LangSmith 클라이언트: push/pull의 핵심 객체예요
# LANGSMITH_API_KEY는 .env에서 자동으로 읽어요
client = Client()

print("LangSmith 클라이언트 초기화 완료")
print(f"프로젝트: {os.environ.get('LANGSMITH_PROJECT')}")

## 1. 프롬프트 생성 및 Push

코드에서 만든 ChatPromptTemplate을 LangSmith Hub에 업로드해요.

> 🎯 **강의 포인트**: `push_prompt()`는 프롬프트의 첫 번째 커밋을 만들어요. 이미 존재하는 프롬프트에 push하면 새 커밋이 추가돼요. URL을 반환하므로 브라우저에서 바로 확인할 수 있어요.

In [ ]:
# ---------------------------------------------------
# 1.1 기본 프롬프트 Push
# ---------------------------------------------------
# ChatPromptTemplate: LangChain의 표준 프롬프트 구조
from langchain.prompts import ChatPromptTemplate

# 고객 이메일 자동 분류 프롬프트 만들기
email_classifier_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 고객 지원 이메일을 분류하는 AI예요.
이메일의 카테고리를 다음 중 하나로 분류해요:
- billing: 결제, 청구, 환불 관련
- technical: 기술 문제, 버그, 오류 관련  
- inquiry: 일반 문의, 제품 정보 요청
- complaint: 불만, 불편사항
- other: 위 항목에 해당하지 않는 경우

카테고리 이름만 소문자로 응답해요."""),
    ("human", "이메일 내용:\n{email_content}")
])

print("프롬프트 생성 완료")
print(f"입력 변수: {email_classifier_prompt.input_variables}")

In [ ]:
# ---------------------------------------------------
# 프롬프트를 LangSmith Hub에 업로드
# ---------------------------------------------------
# push_prompt(): Hub에 업로드하고 URL을 반환해요
# 이름은 영문 소문자와 하이픈만 사용 권장
PROMPT_NAME = "email-classifier-demo"

try:
    prompt_url = client.push_prompt(
        PROMPT_NAME,
        object=email_classifier_prompt,  # ChatPromptTemplate 객체
    )
    print(f"프롬프트 업로드 완료!")
except Exception as e:
    if "Nothing to commit" in str(e) or "Conflict" in str(e):
        print(f"이미 동일한 내용이 저장되어 있어요 (변경사항 없음)")
        prompt_url = f"https://smith.langchain.com/prompts/{PROMPT_NAME}"
    else:
        raise

print(f"URL: {prompt_url}")
print(f"\nLangSmith에서 확인해보세요: {prompt_url}")

In [ ]:
# ---------------------------------------------------
# 1.2 모델을 포함한 체인 Push
# ---------------------------------------------------
# 프롬프트와 모델을 함께 저장하면, pull할 때 모델 설정도 같이 가져올 수 있어요
from langchain_openai import ChatOpenAI

# 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# LCEL로 체인 구성: 프롬프트 → 모델
email_chain = email_classifier_prompt | model

# 체인 전체를 Push (프롬프트 + 모델 설정 포함)
CHAIN_PROMPT_NAME = "email-classifier-with-model-demo"

try:
    chain_url = client.push_prompt(
        CHAIN_PROMPT_NAME,
        object=email_chain  # 체인 전체를 저장
    )
    print(f"체인 업로드 완료!")
    print(f"URL: {chain_url}")
except Exception as e:
    if "Nothing to commit" in str(e) or "Conflict" in str(e):
        print(f"이미 동일한 내용이 저장되어 있어요 (변경사항 없음)")
        chain_url = f"https://smith.langchain.com/prompts/{CHAIN_PROMPT_NAME}"
        print(f"URL: {chain_url}")
    else:
        raise

## 2. 프롬프트 Pull - 다양한 방법으로 불러오기

저장된 프롬프트를 코드에서 불러오는 세 가지 방법을 알아볼게요.

> ⚠️ **자주 하는 실수**: 오래된 패턴인 `langchain.hub.pull()`은 더 이상 권장하지 않아요. 최신 방법은 `client.pull_prompt()`를 사용하는 것이에요.

In [ ]:
# ---------------------------------------------------
# 2.1 최신 버전 Pull (기본)
# ---------------------------------------------------
# pull_prompt("이름"): 가장 최근에 저장된 버전을 가져와요
latest_prompt = client.pull_prompt(PROMPT_NAME)

print("[최신 버전 Pull]")
print(f"타입: {type(latest_prompt)}")
print(f"입력 변수: {latest_prompt.input_variables}")
print(f"\n메시지 구조:")
for i, msg in enumerate(latest_prompt.messages):
    print(f"  {i+1}. {msg.prompt.template[:50]}...")

In [ ]:
# ---------------------------------------------------
# 2.2 Pull한 프롬프트로 실행하기
# ---------------------------------------------------
from langchain.chat_models import ChatOpenAI

# Pull한 프롬프트에 모델 연결
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = latest_prompt | model

# 테스트 이메일로 분류 실행
test_emails = [
    "지난달 결제가 두 번 청구된 것 같아요. 확인해주세요.",
    "앱이 계속 강제종료 됩니다. 오류 코드는 ERR_500 이에요.",
    "프리미엄 플랜의 기능이 궁금해요."
]

print("[이메일 분류 결과]")
for email in test_emails:
    result = chain.invoke({"email_content": email})
    print(f"  이메일: {email[:30]}...")
    print(f"  분류: {result.content}")
    print()

In [ ]:
# ---------------------------------------------------
# 2.3 커밋 해시로 특정 버전 Pull
# ---------------------------------------------------
# 프롬프트가 변경된 후에도 특정 시점의 버전을 정확히 재현할 수 있어요
# (안정성이 중요한 프로덕션 환경에서 유용해요)

# 먼저 프롬프트 정보를 조회해서 최신 커밋 해시를 가져와요
prompt_info = client.get_prompt(PROMPT_NAME)

# 커밋 해시 확인 (앞 8자리만 사용해도 됩니다)
latest_commit_hash = prompt_info.last_commit_hash
print(f"최신 커밋 해시: {latest_commit_hash}")

# 커밋 해시로 정확한 버전 Pull
# 형식: "이름:커밋해시"
versioned_prompt = client.pull_prompt(f"{PROMPT_NAME}:{latest_commit_hash}")
print(f"\n커밋 {latest_commit_hash[:8]}... 버전 Pull 성공")
print(f"타입: {type(versioned_prompt)}")

## 3. 버전 관리 - 커밋과 태그

프롬프트를 개선하면서 체계적으로 버전을 관리하는 방법을 알아볼게요.

> 🎯 **강의 포인트**: 실제 팀에서는 `production` 태그가 붙은 버전만 서비스에서 사용해요. 새 버전을 테스트할 때는 `staging` 태그를 붙이고, 충분히 검증된 후 `production` 태그를 이동해요.

In [ ]:
# ---------------------------------------------------
# 3.1 프롬프트 개선 후 새 버전 Push
# ---------------------------------------------------
# 기존 프롬프트를 개선한 버전 (더 상세한 지시사항 추가)
improved_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 고객 지원 이메일을 정확하게 분류하는 AI예요.

## 분류 카테고리
- billing: 결제, 청구, 환불, 구독, 가격 관련
- technical: 기술 문제, 버그, 오류, 접속 불가, 성능 저하
- inquiry: 제품 정보, 기능 문의, 사용 방법 질문
- complaint: 서비스 불만, 불편사항, 직원 태도 문제
- urgent: 즉각적인 처리가 필요한 긴급 상황
- other: 위 항목에 해당하지 않는 경우

## 응답 형식
카테고리 이름만 소문자로 응답해요. 추가 설명은 하지 않아요.

## 예시
이메일: "카드가 두 번 청구됐어요" → billing
이메일: "앱이 실행 안 돼요" → technical"""),
    ("human", "이메일 내용:\n{email_content}")
])

# 같은 이름으로 Push → 새 커밋이 추가돼요 (이전 버전은 유지)
try:
    updated_url = client.push_prompt(
        PROMPT_NAME,
        object=improved_prompt
    )
    print(f"개선된 버전 Push 완료: {updated_url}")
except Exception as e:
    if "Nothing to commit" in str(e) or "Conflict" in str(e):
        print(f"이미 동일한 내용이 저장되어 있어요 (변경사항 없음)")
        updated_url = f"https://smith.langchain.com/prompts/{PROMPT_NAME}"
        print(f"URL: {updated_url}")
    else:
        raise

In [ ]:
# ---------------------------------------------------
# 3.2 태그 생성 및 관리
# ---------------------------------------------------
# update_prompt(): 기존 프롬프트의 메타데이터(태그 포함)를 업데이트해요

# 현재 최신 버전에 'staging' 태그 부여
# (충분한 테스트 후 'production'으로 승격할 거예요)
prompt_info_updated = client.get_prompt(PROMPT_NAME)
new_commit_hash = prompt_info_updated.last_commit_hash

print(f"새 커밋 해시: {new_commit_hash}")
print(f"이전 커밋 해시: {latest_commit_hash}")
print("\n태그는 LangSmith UI에서 설정하거나 API로 관리할 수 있어요")
print("UI에서: Prompts → 해당 프롬프트 → 버전 히스토리 → 태그 추가")

In [ ]:
# ---------------------------------------------------
# 3.3 태그로 특정 환경의 프롬프트 Pull
# ---------------------------------------------------
# 실제 운영 코드에서는 이렇게 환경별로 프롬프트를 가져와요
# 태그를 변경하는 것만으로 배포 없이 프롬프트를 교체할 수 있어요!

# 주의: 태그는 LangSmith UI나 API에서 수동으로 설정해야 해요
# 아래는 태그가 설정되어 있을 때의 사용 예시예요

def get_prompt_for_environment(prompt_name: str, env: str = "production"):
    """환경에 따라 적절한 버전의 프롬프트를 가져오는 헬퍼 함수예요
    
    Args:
        prompt_name: 프롬프트 이름
        env: 환경 이름 (production, staging, latest)
    """
    if env == "latest":
        # 태그 없이 최신 버전 가져오기
        return client.pull_prompt(prompt_name)
    else:
        # 태그로 특정 환경 버전 가져오기
        # 형식: "이름:태그"
        try:
            return client.pull_prompt(f"{prompt_name}:{env}")
        except Exception:
            # 태그가 없으면 최신 버전 사용
            print(f"'{env}' 태그가 없어요. 최신 버전을 사용해요.")
            return client.pull_prompt(prompt_name)

# 사용 예시
print("[환경별 프롬프트 사용 예시]")
print()
print("# 개발 환경 - 항상 최신 버전 사용")
print(f"dev_prompt = get_prompt_for_environment('{PROMPT_NAME}', 'latest')")
print()
print("# 스테이징 환경 - staging 태그 버전 사용")
print(f"staging_prompt = get_prompt_for_environment('{PROMPT_NAME}', 'staging')")
print()
print("# 프로덕션 환경 - production 태그 버전 사용 (가장 안전한 버전)")
print(f"prod_prompt = get_prompt_for_environment('{PROMPT_NAME}', 'production')")

# 실제 실행
dev_prompt = get_prompt_for_environment(PROMPT_NAME, "latest")
print(f"\n개발 환경 프롬프트 로드 성공: {type(dev_prompt).__name__}")

## 4. 모델 포함 체인 Pull

모델 설정까지 함께 저장하면, 배포 시 코드 변경 없이 프롬프트와 모델을 모두 교체할 수 있어요.

> 💡 **실무 팁**: 모델을 포함해서 Push하면 팀 전체가 동일한 모델 설정을 사용할 수 있어요. "개발자 A는 temperature=0.7, 개발자 B는 temperature=0.3"처럼 설정이 제각각인 문제를 방지해요.

In [ ]:
# ---------------------------------------------------
# 4.1 모델 포함 체인 Pull (include_model=True)
# ---------------------------------------------------
# include_model=True: 프롬프트와 바인딩된 모델까지 함께 가져와요
#
# 주의: langchain_core의 보안 정책으로 인해 파트너 패키지(langchain_openai 등)의
# 클래스 역직렬화가 기본적으로 차단될 수 있어요.
# 이 경우 프롬프트만 Pull하고 모델을 별도로 연결하는 방법을 사용해요.
from langchain_openai import ChatOpenAI

try:
    # 모델이 포함된 체인 Pull 시도
    full_chain = client.pull_prompt(
        CHAIN_PROMPT_NAME,
        include_model=True  # 모델 설정도 함께 가져와요
    )
    print(f"체인 Pull 성공 (모델 포함)")
    print(f"타입: {type(full_chain)}")

except ValueError as e:
    if "Deserialization" in str(e):
        print(f"모델 포함 Pull 실패 (역직렬화 보안 제한)")
        print(f"  원인: {str(e)[:100]}...")
        print()
        print("대안: 프롬프트만 Pull하고 모델을 별도로 연결해요 (권장 방법)")

        # 프롬프트만 Pull (모델 제외)
        prompt_only = client.pull_prompt(CHAIN_PROMPT_NAME, include_model=False)
        model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        full_chain = prompt_only | model
        print(f"  프롬프트 Pull + 모델 연결 성공!")
    else:
        raise

# 바로 실행 가능해요
test_email = "비밀번호를 분실했어요. 어떻게 재설정하나요?"
result = full_chain.invoke({"email_content": test_email})

print(f"\n테스트 이메일: {test_email}")
print(f"분류 결과: {result.content}")

## 5. 프롬프트 목록 조회 및 삭제

Hub에 저장된 프롬프트들을 관리하는 방법을 알아볼게요.

In [ ]:
# ---------------------------------------------------
# 5.1 프롬프트 목록 조회
# ---------------------------------------------------
# list_prompts(): Hub에 저장된 프롬프트 목록을 가져와요
# 반환값은 ListPromptsResponse 객체이고, .repos로 프롬프트 목록에 접근해요

print("[내 프롬프트 목록]")
print("-" * 50)

# 모든 프롬프트 목록
response = client.list_prompts()
all_prompts = response.repos  # Prompt 객체 리스트

for prompt in all_prompts[:10]:  # 최대 10개까지만 출력
    print(f"  이름: {prompt.repo_handle}")
    if hasattr(prompt, 'last_commit_hash') and prompt.last_commit_hash:
        print(f"  최신 커밋: {prompt.last_commit_hash[:8]}")
    print()

print(f"총 {len(all_prompts)}개의 프롬프트가 있어요")

In [ ]:
# ---------------------------------------------------
# 5.2 키워드로 프롬프트 검색
# ---------------------------------------------------
# query 파라미터로 이름에 특정 단어가 포함된 프롬프트만 검색해요

print("[키워드 검색: 'email']")
email_response = client.list_prompts(query="email")
email_prompts = email_response.repos

for prompt in email_prompts:
    print(f"  - {prompt.repo_handle}")

print(f"검색 결과: {len(email_prompts)}개")

In [ ]:
# ---------------------------------------------------
# 5.3 프롬프트 삭제 (실습 후 정리)
# ---------------------------------------------------
# delete_prompt(): 저장된 프롬프트를 완전히 삭제해요 (되돌릴 수 없어요!)

# 실습용으로 만든 프롬프트 삭제
prompts_to_delete = [PROMPT_NAME, CHAIN_PROMPT_NAME]

for prompt_name in prompts_to_delete:
    try:
        client.delete_prompt(prompt_name)
        print(f"삭제 완료: {prompt_name}")
    except Exception as e:
        print(f"삭제 실패: {prompt_name} - {e}")

print("\n실습 프롬프트 정리 완료")

## 6. 템플릿 형식 비교 - F-string vs Mustache

LangSmith는 두 가지 변수 치환 형식을 지원해요.

> 🔑 **핵심 개념**: F-string은 단순 변수 치환에, Mustache는 복잡한 데이터 구조(중첩 객체, 배열 반복)에 사용해요. 대부분의 경우 F-string으로 충분해요.

In [ ]:
# ---------------------------------------------------
# 6.1 F-string 형식 (기본값)
# ---------------------------------------------------
# 단순 변수 치환: {변수명} 형식
# 가장 일반적인 형식이에요
from langchain.prompts import ChatPromptTemplate

fstring_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 {role}의 전문가예요. 항상 {language}로 대답해요."),
        ("human", "{question}")
    ],
    template_format="f-string"  # 기본값 (생략 가능)
)

# F-string 사용 - 단순 변수 치환
result_fstring = fstring_prompt.invoke({
    "role": "Python",
    "language": "한국어",
    "question": "리스트 컴프리헨션이란 무엇인가요?"
})

print("[F-string 형식]")
for msg in result_fstring.messages:
    print(f"  {msg.type}: {msg.content[:60]}...")

In [ ]:
# ---------------------------------------------------
# 6.2 Mustache 형식
# ---------------------------------------------------
# 고급 기능: {{변수}}, {{#배열}}반복{{/배열}}, {{#조건}}조건{{/조건}}
# 복잡한 데이터 구조가 필요할 때 사용해요

mustache_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", """당신은 {{{company}}}의 고객 서비스 담당자예요.

{{#customer_history}}
고객 이전 문의 기록:
{{#queries}}
- {{date}}: {{issue}}
{{/queries}}
{{/customer_history}}

{{^customer_history}}
신규 고객입니다.
{{/customer_history}}"""),
        ("human", "{{{inquiry}}}")
    ],
    template_format="mustache"  # Mustache 형식 지정
)

# Mustache 사용 - 중첩 데이터와 조건 처리
result_mustache = mustache_prompt.invoke({
    "company": "TechShop",
    "inquiry": "배송 상태를 확인하고 싶어요",
    "customer_history": {
        "queries": [
            {"date": "2024-01-10", "issue": "결제 오류"},
            {"date": "2024-02-15", "issue": "상품 교환 요청"}
        ]
    }
})

print("[Mustache 형식]")
for msg in result_mustache.messages:
    print(f"  {msg.type}:")
    print(f"  {msg.content}")
    print()

In [ ]:
# ---------------------------------------------------
# F-string vs Mustache 비교 요약
# ---------------------------------------------------
comparison = {
    "항목": ["변수 치환", "중첩 객체 접근", "배열 반복", "조건 처리", "역조건 (존재 안 할 때)", "사용 난이도"],
    "F-string": ["{name}", "직접 지원 안 됨", "직접 지원 안 됨", "직접 지원 안 됨", "직접 지원 안 됨", "쉬움"],
    "Mustache": ["{{{name}}}", "{{user.name}}", "{{#items}}{{name}}{{/items}}", "{{#flag}}...{{/flag}}", "{{^flag}}...{{/flag}}", "중간"]
}

print("[F-string vs Mustache 비교]")
print(f"{'항목':<20} {'F-string':<25} {'Mustache':<40}")
print("-" * 85)
for i in range(len(comparison["항목"])):
    항목 = comparison["항목"][i]
    fstr = comparison["F-string"][i]
    mst = comparison["Mustache"][i]
    print(f"{항목:<20} {fstr:<25} {mst:<40}")

## 7. OpenAI/Anthropic 형식 변환

LangSmith에서 관리하는 프롬프트를 다른 SDK 형식으로 변환하여 사용할 수 있어요.
이를 통해 LangChain 없이도 LangSmith의 프롬프트 관리 기능을 활용할 수 있어요.

> 💡 **실무 팁**: 일부 팀에서는 LangChain을 사용하지 않고 OpenAI SDK 또는 Anthropic SDK를 직접 사용해요. 이런 경우에도 LangSmith로 프롬프트를 관리하고 변환하여 사용할 수 있어요.

In [ ]:
# ---------------------------------------------------
# 7.1 OpenAI 형식으로 변환
# ---------------------------------------------------
# convert_prompt_to_openai_format(): LangChain 프롬프트 → OpenAI API 형식
from langsmith.client import convert_prompt_to_openai_format
from langchain.prompts import ChatPromptTemplate

# 간단한 테스트 프롬프트 생성
simple_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 유머러스한 AI예요."),
    ("human", "{topic}에 대한 재미있는 이야기를 해줘")
])

# 프롬프트에 변수 채우기 (invoke로 실제 값 넣기)
prompt_value = simple_prompt.invoke({"topic": "Python 프로그래밍"})

# OpenAI API 형식으로 변환
openai_payload = convert_prompt_to_openai_format(prompt_value)

print("[OpenAI API 형식으로 변환됨]")
print(f"타입: {type(openai_payload)}")
print(f"키: {list(openai_payload.keys())}")
print(f"\n메시지 형식:")
for msg in openai_payload.get("messages", []):
    print(f"  role: {msg['role']}")
    print(f"  content: {msg['content'][:50]}...")
    print()

# 실제 OpenAI SDK 사용 시:
# from openai import OpenAI
# oai_client = OpenAI()
# response = oai_client.chat.completions.create(**openai_payload, model="gpt-4o-mini")
print("(OpenAI SDK 직접 호출 시: oai_client.chat.completions.create(**openai_payload, model='gpt-4o-mini'))")

## 8. 프롬프트 캐싱 설정

운영 환경에서는 매 요청마다 LangSmith에서 프롬프트를 가져오면 지연이 발생할 수 있어요.
프롬프트 캐싱을 설정하면 성능을 크게 개선할 수 있어요.

> ⚠️ **자주 하는 실수**: 캐싱 없이 프로덕션에서 `pull_prompt()`를 매 요청마다 호출하면 심각한 성능 저하가 발생해요. 반드시 캐싱을 설정하거나, 앱 시작 시 한 번만 호출하도록 구성해요.

In [ ]:
# ---------------------------------------------------
# 8.1 글로벌 프롬프트 캐시 설정
# ---------------------------------------------------
# configure_global_prompt_cache(): 앱 시작 시 한 번 설정하면 돼요
# 참고: 이 기능은 langsmith >= 0.3.x 이상에서 사용 가능해요
try:
    from langsmith.prompt_cache import configure_global_prompt_cache, prompt_cache_singleton

    configure_global_prompt_cache(
        max_size=100,         # 최대 100개의 프롬프트 캐시
        ttl_seconds=3600,     # 1시간 후 캐시 만료 (프롬프트 변경 반영 주기)
        refresh_interval_seconds=300,  # 5분마다 백그라운드에서 갱신
    )

    print("프롬프트 캐시 설정 완료")
    print(f"  최대 크기: 100개")
    print(f"  TTL: 1시간")
    print(f"  갱신 주기: 5분")
    CACHE_AVAILABLE = True
except ImportError:
    print("langsmith.prompt_cache 모듈을 사용할 수 없어요")
    print("  이 기능은 langsmith 최신 버전에서 지원돼요")
    print("  pip install -U langsmith 로 업데이트하면 사용할 수 있어요")
    print()
    print("대안: 앱 시작 시 pull_prompt()를 한 번 호출하고 변수에 저장해서 재사용해요")
    CACHE_AVAILABLE = False

In [ ]:
# ---------------------------------------------------
# 8.2 캐시 동작 확인
# ---------------------------------------------------
# 같은 프롬프트를 두 번 호출하면 두 번째부터 캐시에서 가져와요
import time

# 실습용 프롬프트를 다시 만들어요 (이전에 삭제했으므로)
test_prompt_for_cache = ChatPromptTemplate.from_messages([
    ("system", "당신은 도움이 되는 AI예요."),
    ("human", "{question}")
])

CACHE_TEST_PROMPT = "cache-test-demo"

try:
    client.push_prompt(CACHE_TEST_PROMPT, object=test_prompt_for_cache)
except Exception:
    pass  # 이미 존재하면 무시

# 첫 번째 호출 (네트워크에서 가져옴)
start = time.time()
prompt1 = client.pull_prompt(CACHE_TEST_PROMPT)
first_call_time = time.time() - start

# 두 번째 호출 (캐시가 있으면 캐시에서 가져옴)
start = time.time()
prompt2 = client.pull_prompt(CACHE_TEST_PROMPT)
second_call_time = time.time() - start

print("[캐시 성능 비교]")
print(f"  첫 번째 호출: {first_call_time:.3f}초")
print(f"  두 번째 호출: {second_call_time:.3f}초")

if CACHE_AVAILABLE:
    print(f"\n[캐시 통계]")
    print(f"  캐시 히트: {prompt_cache_singleton.metrics.hits}")
    print(f"  캐시 미스: {prompt_cache_singleton.metrics.misses}")
else:
    print("\n(캐시 모듈 미설치로 캐시 통계는 확인할 수 없어요)")

# 정리
try:
    client.delete_prompt(CACHE_TEST_PROMPT)
except Exception:
    pass

## 9. 팀 협업 전략

여러 명이 함께 프롬프트를 관리할 때 효과적인 전략을 알아볼게요.

### 환경별 프롬프트 관리 전략

```
[개발] → staging 태그 → [QA 테스트] → production 태그 → [운영]
```

| 태그 | 의미 | 사용 환경 |
|------|------|----------|
| `production` | 검증 완료, 안정적인 버전 | 운영 서버 |
| `staging` | 테스트 중, 검토 대기 | 스테이징 서버 |
| `v1`, `v2` | 메이저 버전 구분 | 히스토리 참조용 |

### 프롬프트 네이밍 컨벤션

```
[서비스]-[기능]-[언어?]
예시:
  customer-service-email-classifier
  rag-query-rewriter-ko
  code-review-feedback-en
```

> 🎯 **강의 포인트**: 프롬프트를 Git처럼 관리하면 "언제, 누가, 왜 프롬프트를 변경했는지" 추적할 수 있어요. 모델 성능 문제가 생겼을 때 특정 프롬프트 변경이 원인인지 빠르게 확인할 수 있어요.

In [ ]:
# ============================================================
# TODO: 나만의 프롬프트 Hub 관리 실습
#
# 1. 아래 실습 프롬프트를 목적에 맞게 수정해보세요
#    (이메일 분류 → 다른 용도로 변경: 감정 분석, 요약, 번역 등)
# 2. 수정한 프롬프트를 Hub에 Push하세요
# 3. 프롬프트를 개선하고 다시 Push해서 버전 히스토리를 만들어보세요
# 4. LangSmith UI에서 버전 히스토리를 확인하고 staging 태그를 붙여보세요
# 힌트: client.push_prompt()로 같은 이름으로 두 번 push하면 두 개의 커밋이 생겨요
# 예상 결과: LangSmith UI의 Prompts 섹션에서 버전 히스토리 확인 가능
# ============================================================

from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# TODO: 이 프롬프트를 원하는 용도로 수정해보세요!
MY_PROMPT_NAME = "my-practice-prompt"  # 원하는 이름으로 변경

# 버전 1: 기본 프롬프트
my_prompt_v1 = ChatPromptTemplate.from_messages([
    ("system", "당신은 도움이 되는 AI예요."),  # 여기를 구체적으로 수정하세요
    ("human", "{input}")
])

# 버전 1 Push
try:
    url_v1 = client.push_prompt(MY_PROMPT_NAME, object=my_prompt_v1)
    print(f"버전 1 Push 완료: {url_v1}")
except Exception as e:
    if "Nothing to commit" in str(e) or "Conflict" in str(e):
        url_v1 = f"https://smith.langchain.com/prompts/{MY_PROMPT_NAME}"
        print(f"이미 동일한 내용이 저장되어 있어요: {url_v1}")
    else:
        raise

# TODO: 버전 2를 만들어 개선된 프롬프트를 Push해보세요
# my_prompt_v2 = ChatPromptTemplate.from_messages([...])
# url_v2 = client.push_prompt(MY_PROMPT_NAME, object=my_prompt_v2)

print("\nLangSmith UI에서 버전 히스토리를 확인해보세요!")
print(f"URL: {url_v1}")

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **push_prompt()**: ChatPromptTemplate을 LangSmith Hub에 업로드 (URL 반환)
- **모델 포함 Push**: `프롬프트 | 모델` 체인 전체를 저장하면 모델 설정도 함께 관리
- **pull_prompt() 3가지 방법**:
  - `pull_prompt("name")`: 최신 버전
  - `pull_prompt("name:production")`: 태그로 특정 버전
  - `pull_prompt("name:a3f92c1d")`: 커밋 해시로 정확한 버전
- **버전 관리**: 저장 시 자동 커밋 생성, 수동으로 태그 지정 (`production`, `staging`)
- **F-string vs Mustache**: 단순 변수는 F-string, 복잡한 데이터(배열/조건)는 Mustache
- **형식 변환**: `convert_prompt_to_openai_format()`으로 다른 SDK와 호환
- **프롬프트 캐싱**: `configure_global_prompt_cache()`로 운영 환경 성능 개선

## 다음 단계

이 노트북은 LangSmith 시리즈의 마지막 노트북이에요.

다음 단계로 **07_Evaluation** 과정에서 LangSmith의 평가(Evaluation) 기능을 학습해요.
데이터셋, 평가자(Evaluator), 실험(Experiment)을 활용하여 에이전트의 성능을 체계적으로 측정하는 방법을 배울 거예요.